# 03 · Ouro dimensional, LGPD e consultas do MVP (requisitos 07 e 08)

**Grão do fato:** `ouro_fato_chuva` tem **uma linha por município por hora**.

```
                ouro_dim_tempo (id_tempo)
                        |
ouro_dim_municipio --- ouro_fato_chuva --- ouro_dim_intensidade
 (codigo_ibge)          (métricas)          (id_intensidade)
                        |
               ouro_relatos (LGPD: hash/máscara) -> mesmas chaves de tempo e município
```
Intensidade da chuva por hora segue a classificação usual de taxa de precipitação:
fraca < 2,5 mm/h · moderada 2,5–7,6 · **forte 7,6–50** · **violenta ≥ 50**. Forte e violenta disparam **alerta**.

In [0]:
SET TIME ZONE 'America/Sao_Paulo';

key,value
spark.sql.session.timeZone,America/Sao_Paulo


## 1. Dimensões

In [0]:
CREATE OR REPLACE TABLE workspace.gs.ouro_dim_tempo AS
SELECT DISTINCT
       CAST(date_format(data_hora, 'yyyyMMddHH') AS BIGINT) AS id_tempo,
       data_hora,
       to_date(data_hora)                                   AS data,
       hour(data_hora)                                      AS hora,
       dayofweek(data_hora)                                 AS dia_semana_num,
       CASE dayofweek(data_hora) WHEN 1 THEN 'domingo' WHEN 2 THEN 'segunda' WHEN 3 THEN 'terça'
            WHEN 4 THEN 'quarta' WHEN 5 THEN 'quinta' WHEN 6 THEN 'sexta' ELSE 'sábado' END AS dia_semana,
       dayofweek(data_hora) IN (1, 7)                       AS fim_de_semana,
       month(data_hora)                                     AS mes,
       year(data_hora)                                      AS ano,
       CASE WHEN hour(data_hora) < 6  THEN 'madrugada'
            WHEN hour(data_hora) < 12 THEN 'manhã'
            WHEN hour(data_hora) < 18 THEN 'tarde'
            ELSE 'noite' END                                AS periodo_dia
FROM workspace.gs.prata_clima;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE workspace.gs.ouro_dim_municipio AS
SELECT i.codigo_ibge, i.municipio, i.uf, i.populacao, i.ano_referencia AS ano_populacao,
       c.latitude, c.longitude
FROM workspace.gs.prata_ibge i
JOIN (SELECT codigo_ibge, MAX(latitude) AS latitude, MAX(longitude) AS longitude
      FROM workspace.gs.prata_clima GROUP BY codigo_ibge) c
  ON i.codigo_ibge = c.codigo_ibge;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE workspace.gs.ouro_dim_intensidade AS
SELECT * FROM VALUES
  (0, 'sem chuva', 0.0,   0.1,     false),
  (1, 'fraca',     0.1,   2.5,     false),
  (2, 'moderada',  2.5,   7.6,     false),
  (3, 'forte',     7.6,   50.0,    true),
  (4, 'violenta',  50.0,  100000.0, true)
AS t(id_intensidade, intensidade, mm_h_min, mm_h_max, gera_alerta);

num_affected_rows,num_inserted_rows


## 2. Fato: uma linha por município por hora

In [0]:
CREATE OR REPLACE TABLE workspace.gs.ouro_fato_chuva AS
WITH relatos_hora AS (
  SELECT codigo_ibge,
         CAST(date_format(date_trunc('HOUR', data_hora), 'yyyyMMddHH') AS BIGINT) AS id_tempo,
         COUNT(*)                                                       AS qtd_relatos,
         SUM(CASE WHEN nivel_agua IN ('joelho', 'cintura') THEN 1 ELSE 0 END) AS qtd_relatos_agua_alta
  FROM workspace.gs.prata_relatos
  GROUP BY ALL
)
SELECT CAST(date_format(c.data_hora, 'yyyyMMddHH') AS BIGINT) AS id_tempo,
       c.codigo_ibge,
       i.id_intensidade,
       c.chuva_mm,
       c.temperatura_c,
       c.umidade_pct,
       c.rajada_kmh,
       COALESCE(r.qtd_relatos, 0)            AS qtd_relatos,
       COALESCE(r.qtd_relatos_agua_alta, 0)  AS qtd_relatos_agua_alta,
       i.gera_alerta                         AS em_alerta
FROM workspace.gs.prata_clima c
JOIN workspace.gs.ouro_dim_intensidade i
  ON c.chuva_mm >= i.mm_h_min AND c.chuva_mm < i.mm_h_max
LEFT JOIN relatos_hora r
  ON r.codigo_ibge = c.codigo_ibge
 AND r.id_tempo = CAST(date_format(c.data_hora, 'yyyyMMddHH') AS BIGINT);

num_affected_rows,num_inserted_rows


In [0]:
-- PRINT: integridade. Chaves das dimensões únicas e fato batendo com a Prata.
SELECT 'dim_tempo: chaves duplicadas'      AS verificacao, COUNT(*) - COUNT(DISTINCT id_tempo)      AS resultado, '0 esperado' AS esperado FROM workspace.gs.ouro_dim_tempo
UNION ALL SELECT 'dim_municipio: chaves duplicadas', COUNT(*) - COUNT(DISTINCT codigo_ibge), '0 esperado' FROM workspace.gs.ouro_dim_municipio
UNION ALL SELECT 'dim_intensidade: chaves duplicadas', COUNT(*) - COUNT(DISTINCT id_intensidade), '0 esperado' FROM workspace.gs.ouro_dim_intensidade
UNION ALL SELECT 'linhas fato - linhas prata_clima',
  (SELECT COUNT(*) FROM workspace.gs.ouro_fato_chuva) - (SELECT COUNT(*) FROM workspace.gs.prata_clima), '0 esperado'
UNION ALL SELECT 'soma chuva fato - soma chuva prata',
  ROUND((SELECT SUM(chuva_mm) FROM workspace.gs.ouro_fato_chuva) - (SELECT SUM(chuva_mm) FROM workspace.gs.prata_clima), 6), '0 esperado'
UNION ALL SELECT 'relatos no fato - relatos na prata',
  (SELECT SUM(qtd_relatos) FROM workspace.gs.ouro_fato_chuva) - (SELECT COUNT(*) FROM workspace.gs.prata_relatos),
  '0 ou negativo (relatos de horas ainda sem clima)'
UNION ALL SELECT 'fato sem município na dimensão',
  (SELECT COUNT(*) FROM workspace.gs.ouro_fato_chuva f LEFT ANTI JOIN workspace.gs.ouro_dim_municipio m USING (codigo_ibge)), '0 esperado';

verificacao,resultado,esperado
dim_tempo: chaves duplicadas,0.0,0 esperado
dim_municipio: chaves duplicadas,0.0,0 esperado
dim_intensidade: chaves duplicadas,0.0,0 esperado
linhas fato - linhas prata_clima,0.0,0 esperado
soma chuva fato - soma chuva prata,0.0,0 esperado
relatos no fato - relatos na prata,0.0,0 ou negativo (relatos de horas ainda sem clima)
fato sem município na dimensão,0.0,0 esperado


## 3. LGPD: relatos protegidos na Ouro
| Dado pessoal | Tratamento na Ouro |
|---|---|
| nome | **não levado** (minimização: o MVP não precisa) |
| e-mail | **hash** `sha2(lower(email) + sal, 256)`; sal em `_controle/sal.txt`, fora do código |
| telefone | **máscara** `(11) *****-1234` |
| latitude/longitude exatas | **arredondadas a 3 casas** (~100 m): mostra a rua, não a casa |

In [0]:
%python
from pyspark.sql import functions as F

sal = dbutils.fs.head("/Volumes/workspace/gs/chegada/_controle/sal.txt").strip()

(spark.table("workspace.gs.prata_relatos")
   .select(
       "id_relato",
       F.date_format(F.date_trunc("hour", "data_hora"), "yyyyMMddHH").cast("bigint").alias("id_tempo"),
       "codigo_ibge",
       F.sha2(F.concat(F.lower("email"), F.lit(sal)), 256).alias("email_hash"),
       F.concat(F.substring("telefone", 1, 5), F.lit("*****-"), F.substring("telefone", -4, 4)).alias("telefone_mascarado"),
       F.round("latitude", 3).alias("latitude_aprox"),
       F.round("longitude", 3).alias("longitude_aprox"),
       "nivel_agua")
   .write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("workspace.gs.ouro_relatos"))

display(spark.table("workspace.gs.ouro_relatos").limit(5))   # PRINT: dados protegidos

id_relato,id_tempo,codigo_ibge,email_hash,telefone_mascarado,latitude_aprox,longitude_aprox,nivel_agua
0001cc88-5d04-42fb-71e3-f2500900ecd8,2026102505,3548807,31bc82072a63916adce97d4c7d39a82c91ee5bc51b6462a8b40fde9dbf6f85fe,(11) *****-8776,-23.602,-46.589,canela
001340cd-c7db-14c5-bd07-45dbbc6767b7,2026100118,3518800,75632a12141afd8bac3b105111c3672c69cbf1c620ec874c11f915e2771ed212,(11) *****-4559,-23.434,-46.588,canela
00280a3b-fd40-a2c5-c5e6-6bbec03066be,2026100704,3513801,cfcb789f9bf9033066f2bb6b68f9fef0189132e25717ba115c107ebd520ea594,(11) *****-7947,-23.654,-46.597,calcada
00290e7a-81ad-8a6a-759a-bd4a50f0eb5c,2026100917,3550308,93a63eca09ef221224c5a3e95a7a39c05040dcce5a7039e9102a3bf3295eda09,(11) *****-1140,-23.544,-46.593,calcada
0061b889-ed57-77b8-0ae0-7484f34fd8eb,2026102919,3548807,4aae7df449a80f50474ac8c67b2f858fc4f672eb1db8d35ba4e127de9f9220d6,(11) *****-4973,-23.578,-46.573,calcada


## 4. As 5 consultas do MVP
"Hoje" = a data mais recente coletada (a coleta do dia traz a previsão das 24 horas).

In [0]:
-- Q1 · "Vai chover forte hoje no meu município? A partir de que horas?"
WITH hoje AS (SELECT MAX(t.data) AS d FROM workspace.gs.ouro_fato_chuva f JOIN workspace.gs.ouro_dim_tempo t USING (id_tempo))
SELECT m.municipio,
       ROUND(MAX(f.chuva_mm), 1)                                AS pico_mm_h,
       max_by(i.intensidade, f.chuva_mm)                        AS intensidade_maxima,
       SUM(CASE WHEN f.em_alerta THEN 1 ELSE 0 END)             AS horas_em_alerta,
       MIN(CASE WHEN f.em_alerta THEN t.hora END)               AS alerta_a_partir_das,
       ROUND(SUM(f.chuva_mm), 1)                                AS chuva_total_dia_mm
FROM workspace.gs.ouro_fato_chuva f
JOIN workspace.gs.ouro_dim_tempo t        USING (id_tempo)
JOIN workspace.gs.ouro_dim_municipio m    USING (codigo_ibge)
JOIN workspace.gs.ouro_dim_intensidade i  USING (id_intensidade)
WHERE t.data = (SELECT d FROM hoje)
GROUP BY m.municipio
ORDER BY horas_em_alerta DESC, pico_mm_h DESC;

municipio,pico_mm_h,intensidade_maxima,horas_em_alerta,alerta_a_partir_das,chuva_total_dia_mm
Guarulhos,2.4,fraca,0,null,6.2
São Bernardo do Campo,2.2,fraca,0,null,6.2
São Paulo,2.1,fraca,0,null,6.2
Barueri,1.9,fraca,0,null,5.7
Mauá,1.9,fraca,0,null,5.2
São Caetano do Sul,1.9,fraca,0,null,6.0
Osasco,1.6,fraca,0,null,4.5
Diadema,1.5,fraca,0,null,4.9
Santo André,1.2,fraca,0,null,5.6
Taboão da Serra,1.2,fraca,0,null,5.0


Q1 · “Vai chover forte hoje no meu município?”

Em 05/11, nenhum dos 10 municípios entrou em alerta: a chuva foi fraca em todos, com pico de 2,4 mm/h em Guarulhos e acumulado de até 6,2 mm no dia.
 Na tela do MVP, o cidadão veria “sem alerta hoje”, com a previsão de chuva fraca hora a hora.

In [0]:
-- Q2 · "Quais municípios tiveram mais horas de chuva forte nos últimos 30 dias?" (ranking de risco)
WITH ref AS (SELECT MAX(t.data) AS d FROM workspace.gs.ouro_fato_chuva f JOIN workspace.gs.ouro_dim_tempo t USING (id_tempo))
SELECT RANK() OVER (ORDER BY SUM(CASE WHEN f.em_alerta THEN 1 ELSE 0 END) DESC) AS posicao,
       m.municipio,
       SUM(CASE WHEN f.em_alerta THEN 1 ELSE 0 END)           AS horas_em_alerta,
       COUNT(DISTINCT CASE WHEN f.em_alerta THEN t.data END)  AS dias_com_alerta,
       ROUND(SUM(f.chuva_mm), 1)                              AS chuva_acumulada_mm,
       ROUND(MAX(f.chuva_mm), 1)                              AS maior_chuva_mm_h
FROM workspace.gs.ouro_fato_chuva f
JOIN workspace.gs.ouro_dim_tempo t     USING (id_tempo)
JOIN workspace.gs.ouro_dim_municipio m USING (codigo_ibge)
WHERE t.data > (SELECT date_sub(d, 30) FROM ref)
GROUP BY m.municipio
ORDER BY posicao, chuva_acumulada_mm DESC;

posicao,municipio,horas_em_alerta,dias_com_alerta,chuva_acumulada_mm,maior_chuva_mm_h
1,São Paulo,1,1,219.2,11.0
1,Guarulhos,1,1,219.2,9.0
1,Osasco,1,1,213.8,8.7
1,São Caetano do Sul,1,1,211.4,10.6
1,Taboão da Serra,1,1,210.7,8.1
1,Santo André,1,1,186.7,7.6
1,Mauá,1,1,182.8,8.0
8,Barueri,0,0,199.1,6.7
8,Diadema,0,0,188.9,7.2
8,São Bernardo do Campo,0,0,182.8,7.1


Q2 · Ranking de risco nos últimos 30 dias

7 dos 10 municípios tiveram 1 hora de chuva forte (≥ 7,6 mm/h) no período, todas no mesmo episódio. São Paulo registrou o maior pico (11 mm/h) e, junto com Guarulhos, o maior acumulado (219 mm). Barueri, Diadema e São Bernardo não chegaram ao nível de alerta, apesar de acumulados próximos de 190 mm, o que mostra que o risco vem da intensidade concentrada e não do total do mês.

In [0]:
-- Q3 · "Em que horário do dia a chuva forte costuma cair?" (para planejar o deslocamento)
SELECT t.periodo_dia,
       t.hora,
       SUM(CASE WHEN f.em_alerta THEN 1 ELSE 0 END)                         AS horas_em_alerta,
       ROUND(100 * AVG(CASE WHEN f.em_alerta THEN 1.0 ELSE 0.0 END), 2)     AS pct_horas_em_alerta,
       ROUND(AVG(f.chuva_mm), 2)                                            AS chuva_media_mm_h
FROM workspace.gs.ouro_fato_chuva f
JOIN workspace.gs.ouro_dim_tempo t USING (id_tempo)
GROUP BY t.periodo_dia, t.hora
ORDER BY t.hora;

periodo_dia,hora,horas_em_alerta,pct_horas_em_alerta,chuva_media_mm_h
madrugada,0,2,0.56,0.36
madrugada,1,2,0.56,0.34
madrugada,2,0,0.00,0.26
madrugada,3,0,0.00,0.23
madrugada,4,0,0.00,0.13
madrugada,5,0,0.00,0.2
manhã,6,0,0.00,0.22
manhã,7,0,0.00,0.2
manhã,8,0,0.00,0.17
manhã,9,0,0.00,0.16


Q3 · Em que horário a chuva forte cai

As horas de alerta se concentraram no início da tarde (14h) e na madrugada (0h e 1h). As manhãs não tiveram nenhuma hora de alerta. Para o cidadão, a recomendação é planejar deslocamentos longos pela manhã.

In [0]:
-- Q4 · "Onde os cidadãos mais relatam alagamento, e isso bate com a chuva medida?"
SELECT m.municipio,
       SUM(f.qtd_relatos)                                                      AS relatos,
       SUM(CASE WHEN f.em_alerta THEN f.qtd_relatos ELSE 0 END)                AS relatos_em_hora_de_alerta,
       SUM(CASE WHEN f.chuva_mm = 0 THEN f.qtd_relatos ELSE 0 END)             AS relatos_sem_chuva_na_hora,
       ROUND(100 * SUM(CASE WHEN f.chuva_mm >= 2.5 THEN f.qtd_relatos ELSE 0 END)
                 / NULLIF(SUM(f.qtd_relatos), 0), 1)                           AS pct_relatos_com_chuva_moderada_ou_mais,
       ROUND(SUM(f.qtd_relatos) / NULLIF(SUM(f.chuva_mm), 0), 2)               AS relatos_por_mm_de_chuva,
       ROUND(corr(f.chuva_mm, f.qtd_relatos), 2)                               AS correlacao_chuva_relatos
FROM workspace.gs.ouro_fato_chuva f
JOIN workspace.gs.ouro_dim_municipio m USING (codigo_ibge)
GROUP BY m.municipio
ORDER BY relatos DESC;

municipio,relatos,relatos_em_hora_de_alerta,relatos_sem_chuva_na_hora,pct_relatos_com_chuva_moderada_ou_mais,relatos_por_mm_de_chuva,correlacao_chuva_relatos
São Caetano do Sul,845,15,19,30.8,3.23,0.84
Taboão da Serra,781,18,22,35.2,3.06,0.91
Osasco,771,8,22,44.0,2.95,0.87
São Paulo,708,30,26,37.4,2.58,0.91
Diadema,648,0,23,22.1,2.83,0.86
São Bernardo do Campo,400,0,12,18.5,1.78,0.87
Guarulhos,376,18,14,37.8,1.37,0.89
Barueri,342,0,14,35.7,1.4,0.88
Santo André,208,7,16,25.0,0.89,0.85
Mauá,183,9,12,27.9,0.76,0.83


Q4 · Relatos de alagamento × chuva medida

A correlação entre chuva e relatos ficou entre 0,83 e 0,91 em todos os municípios: os cidadãos relatam alagamento quando chove, o que valida a fonte. Comparando relatos por mm de chuva, São Caetano do Sul (3,23) e Taboão da Serra (3,06) alagam três a quatro vezes mais que Mauá (0,76) e Santo André (0,89) para a mesma chuva, o que indica pior drenagem. Esse ranking orienta onde a prefeitura deve priorizar obras.

In [0]:
-- Q5 · IMPACTO · "Quantas pessoas o alerta protege?"
-- pessoas-dia sob alerta = soma, dia a dia, da população dos municípios que tiveram ao menos 1 hora de alerta;
-- cobertura = % dos relatos de água alta (joelho/cintura) que aconteceram numa hora em que o app já estava em alerta.
WITH dia_municipio AS (
  SELECT t.data, f.codigo_ibge, MAX(CASE WHEN f.em_alerta THEN 1 ELSE 0 END) AS teve_alerta
  FROM workspace.gs.ouro_fato_chuva f JOIN workspace.gs.ouro_dim_tempo t USING (id_tempo)
  GROUP BY t.data, f.codigo_ibge
),
populacao AS (
  SELECT COUNT(DISTINCT d.data)                                  AS dias_analisados,
         COUNT(DISTINCT CASE WHEN d.teve_alerta = 1 THEN d.data END) AS dias_com_algum_alerta,
         SUM(CASE WHEN d.teve_alerta = 1 THEN m.populacao ELSE 0 END) AS pessoas_dia_sob_alerta,
         MAX(m_total.total)                                      AS populacao_monitorada
  FROM dia_municipio d
  JOIN workspace.gs.ouro_dim_municipio m USING (codigo_ibge)
  CROSS JOIN (SELECT SUM(populacao) AS total FROM workspace.gs.ouro_dim_municipio) m_total
),
cobertura AS (
  SELECT SUM(qtd_relatos_agua_alta)                                   AS relatos_agua_alta,
         SUM(CASE WHEN em_alerta THEN qtd_relatos_agua_alta ELSE 0 END) AS relatos_agua_alta_com_alerta_ativo
  FROM workspace.gs.ouro_fato_chuva
)
SELECT p.populacao_monitorada,
       p.dias_analisados,
       p.dias_com_algum_alerta,
       p.pessoas_dia_sob_alerta,
       ROUND(p.pessoas_dia_sob_alerta / NULLIF(p.dias_com_algum_alerta, 0)) AS media_pessoas_avisadas_por_dia_de_alerta,
       c.relatos_agua_alta,
       c.relatos_agua_alta_com_alerta_ativo,
       ROUND(100 * c.relatos_agua_alta_com_alerta_ativo / NULLIF(c.relatos_agua_alta, 0), 1) AS pct_cobertura_do_alerta
FROM populacao p CROSS JOIN cobertura c;

populacao_monitorada,dias_analisados,dias_com_algum_alerta,pessoas_dia_sob_alerta,media_pessoas_avisadas_por_dia_de_alerta,relatos_agua_alta,relatos_agua_alta_com_alerta_ativo,pct_cobertura_do_alerta
16599201,36,4,16788794,4197199.0,683,64,9.4


Q5 · Impacto: quantas pessoas o alerta protege

Os 10 municípios somam 16,6 milhões de habitantes. Em 36 dias, houve 4 dias com alerta, que somaram 16,8 milhões de pessoas-dia avisadas, uma média de 4,2 milhões de pessoas por dia de alerta. Dos 683 relatos de água alta (joelho ou cintura), só 64 (9,4%) aconteceram com o alerta ativo. Ou seja, a maioria dos alagamentos graves ocorreu com chuva moderada, abaixo do limite de 7,6 mm/h. Para a próxima versão do produto, a recomendação é também alertar a partir de chuva moderada (2,5 mm/h) nos municípios com pior drenagem da Q4.

## 5. Contrato com o MVP (requisito 08)
O MVP consome a view **`ouro_vw_mvp_alertas`**, exportada como `mvp_alertas.csv` e `mvp_alertas.json` em
`/Volumes/workspace/gs/chegada/_export/`. Atualização: diária, depois da carga dos lotes. O MVP **não** recebe dado pessoal.

In [0]:
CREATE OR REPLACE VIEW workspace.gs.ouro_vw_mvp_alertas AS
SELECT m.codigo_ibge, m.municipio, m.populacao, m.latitude, m.longitude,
       t.data_hora, t.data, t.hora, t.periodo_dia,
       f.chuva_mm, i.intensidade, f.em_alerta,
       f.temperatura_c, f.rajada_kmh, f.qtd_relatos, f.qtd_relatos_agua_alta
FROM workspace.gs.ouro_fato_chuva f
JOIN workspace.gs.ouro_dim_tempo t        USING (id_tempo)
JOIN workspace.gs.ouro_dim_municipio m    USING (codigo_ibge)
JOIN workspace.gs.ouro_dim_intensidade i  USING (id_intensidade);

DESCRIBE workspace.gs.ouro_vw_mvp_alertas;

col_name,data_type,comment
codigo_ibge,int,null
municipio,string,null
populacao,bigint,null
latitude,double,null
longitude,double,null
data_hora,timestamp,null
data,date,null
hora,int,null
periodo_dia,string,null
chuva_mm,double,null


In [0]:
%python
# Extrato para o MVP: baixe pela interface (Catalog > Volume > _export) e entregue para quem faz o app
pdf = spark.table("workspace.gs.ouro_vw_mvp_alertas").orderBy("data_hora", "municipio").toPandas()
pdf["data_hora"] = pdf["data_hora"].astype(str)
pdf["data"] = pdf["data"].astype(str)
pdf.to_csv("/Volumes/workspace/gs/chegada/_export/mvp_alertas.csv", index=False)
pdf.to_json("/Volumes/workspace/gs/chegada/_export/mvp_alertas.json", orient="records", force_ascii=False)
print(f"{len(pdf)} linhas exportadas")
display(dbutils.fs.ls("/Volumes/workspace/gs/chegada/_export"))

8640 linhas exportadas


path,name,size,modificationTime
dbfs:/Volumes/workspace/gs/chegada/_export/mvp_alertas.csv,mvp_alertas.csv,1059906,1793917020000
dbfs:/Volumes/workspace/gs/chegada/_export/mvp_alertas.json,mvp_alertas.json,2934614,1793917020000
